# Complex CAPTCHA Features

## Hypothesis

Template-mining features from blocks **D1-D4** and SSL sequence embeddings from blocks **E1-E8** should improve separation between human and bot sessions beyond the original `competition_main.ipynb` feature family. The primary validation metric for model comparison and ensemble weighting is `roc_auc_score(y_true, y_pred, max_fpr=0.1)`.

## Notes

- The notebook keeps the orchestration readable and delegates heavy reusable logic into `src/complex_feature_pipeline.py`.
- The final artifact is written to `outputs/submit_complex.csv`.
- Intermediate caches are stored under `outputs/complex_cache/` so repeated runs can resume quickly.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import torch

from src.complex_feature_pipeline import run_full_complex_pipeline

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TRAIN_PATH = ROOT / 'data' / 'train.parquet'
TEST_PATH = ROOT / 'data' / 'test.parquet'
UNLABELED_PATH = ROOT / 'data' / 'unlabelled.parquet'
CACHE_DIR = ROOT / 'outputs' / 'complex_cache'
SUBMISSION_PATH = ROOT / 'outputs' / 'submit_complex.csv'
METRICS_PATH = ROOT / 'outputs' / 'complex_metrics.json'

print({'device': DEVICE, 'submission_path': str(SUBMISSION_PATH), 'metrics_path': str(METRICS_PATH)})

In [ ]:
results = run_full_complex_pipeline(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    unlabeled_path=UNLABELED_PATH,
    cache_dir=CACHE_DIR,
    submission_path=SUBMISSION_PATH,
    metrics_path=METRICS_PATH,
    device=DEVICE,
)
results

In [ ]:
metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
pd.DataFrame(metrics['model_scores']).T.sort_values('pauc_01', ascending=False)

In [ ]:
pd.Series(metrics['weights']).sort_values(ascending=False)

## Conclusion

After execution, use the tables above as the experiment summary. The key number to compare across models is **pAUC@0.1** (`roc_auc_score(..., max_fpr=0.1)`).